In [13]:
!pip install pandas numpy scikit-learn matplotlib seaborn 


In [14]:
!pip install xgboost

In [15]:
# ====================================================
# 🔋 ELECTRIC VEHICLE ML PROJECT
# Predict Battery Health (%) & Monthly Charging Cost (USD)
# Using Supervised Machine Learning (Regression)
# ====================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# ----- Step 1: Load Dataset -----
file_path = "electric_vehicle_analytics (2).csv"  # <- use your dataset path
df = pd.read_csv("EVproject/electric_vehicle_analytics (2).csv")

print("✅ Dataset Loaded Successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ----- Step 2: Clean Data -----
df = df.dropna()

# Encode categorical columns
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# ----- Step 3: Define Features & Target -----
X = df.drop(columns=['Battery_Health_%', 'Monthly_Charging_Cost_USD'], errors='ignore')
y = df[['Battery_Health_%', 'Monthly_Charging_Cost_USD']]

# ----- Step 4: Scale Data -----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ----- Step 5: Train-Test Split -----
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ----- Step 6: Train Multi-Output XGBoost Model -----
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

multi_model = MultiOutputRegressor(xgb)
multi_model.fit(X_train, y_train)

# ----- Step 7: Evaluate Model -----
y_pred = multi_model.predict(X_test)

# Separate predictions
y_pred_health = y_pred[:, 0]
y_pred_cost = y_pred[:, 1]

# Metrics
mae_health = mean_absolute_error(y_test['Battery_Health_%'], y_pred_health)
r2_health = r2_score(y_test['Battery_Health_%'], y_pred_health)

mae_cost = mean_absolute_error(y_test['Monthly_Charging_Cost_USD'], y_pred_cost)
r2_cost = r2_score(y_test['Monthly_Charging_Cost_USD'], y_pred_cost)

print("\n📊 MODEL PERFORMANCE")
print("Battery Health  → MAE:", round(mae_health, 2), "| R²:", round(r2_health, 3))
print("Charging Cost   → MAE:", round(mae_cost, 2), "| R²:", round(r2_cost, 3))

# ----- Step 8: Example Prediction -----
sample = pd.DataFrame({
    'Battery_Capacity_kWh': [75],
    'Range_km': [350],
    'Charging_Cycles': [800],
    'Avg_Temperature': [30],
    'Daily_Usage_km': [60],
    'Battery_Age_Years': [3],
    'Charging_Frequency': [4],
    'Region': [1],
    'Manufacturer': [2]
})

# Match sample columns with training features
sample = sample.reindex(columns=X.columns, fill_value=0)
sample_scaled = scaler.transform(sample)

pred = multi_model.predict(sample_scaled)
print("\n🔍 Example Prediction:")
print(f"Predicted Battery Health: {pred[0][0]:.2f}%")
print(f"Predicted Monthly Charging Cost: ${pred[0][1]:.2f}")

# ----- Step 9: Save Model -----
import joblib
joblib.dump(multi_model, "ev_multi_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("\n💾 Models saved successfully!")


✅ Dataset Loaded Successfully!
Shape: (3000, 25)
Columns: ['Vehicle_ID', 'Make', 'Model', 'Year', 'Region', 'Vehicle_Type', 'Battery_Capacity_kWh', 'Battery_Health_%', 'Range_km', 'Charging_Power_kW', 'Charging_Time_hr', 'Charge_Cycles', 'Energy_Consumption_kWh_per_100km', 'Mileage_km', 'Avg_Speed_kmh', 'Max_Speed_kmh', 'Acceleration_0_100_kmh_sec', 'Temperature_C', 'Usage_Type', 'CO2_Saved_tons', 'Maintenance_Cost_USD', 'Insurance_Cost_USD', 'Electricity_Cost_USD_per_kWh', 'Monthly_Charging_Cost_USD', 'Resale_Value_USD']

📊 MODEL PERFORMANCE
Battery Health  → MAE: 7.85 | R²: -0.086
Charging Cost   → MAE: 15.06 | R²: 0.995

🔍 Example Prediction:
Predicted Battery Health: 88.76%
Predicted Monthly Charging Cost: $15.86

💾 Models saved successfully!


In [ ]:
print "Hello"